# `numpy`中的一元多项式(`polynomial.Polynomial`)对象, 以及支持的运算

In [1]:
#以别名形式导入numpy. 
import numpy as np; 
#导入与ndarray有关的数据类型. 
from numpy import int8, int16, int32, int64; 
from numpy import uint8, uint16, uint32, uint64; 
from numpy import float16, float32, float64; 
from numpy import complex64, complex128; 

In [2]:
from numpy import polynomial as nppn

## `numpy.polynomial.Polynomial`对象的构造
* 注意: `np.polynomial.Polynomial`只能用于定义标量的多项式, **不能**用于定义**矩阵多项式**. 

### 使用$x^n$项系数列表构造多项式
用法: 
```python
px = np.polynomial.Polynomial(iter_coef)
```
* `iter_coef` 包含多项式系数的迭代器
* 支持的迭代器类型: 
    * `list`, `tuple`, `numpy.ndarray`
    * 迭代器中不能内嵌迭代器; `numpy.ndarray`的`ndim`属性必须为1
* 迭代器内的所有元素应当为数值型对象, 否则将报错`ValueError`
    * Python内建的, 或`numpy`模块中定义的各类`int`, `float`, `complex`; 
    * `Decimal`, `Fraction`
    * 当迭代器为`list`或`tuple`时, 其中的元素类型不需要完全一致
* 系数按照自变量**次数从低到高**排列
    * 下标为`i`的元素表示`i`次项的系数
    * 除最高次项以外的其他任意项不存在时, 对应的系数为0, 必须列出

In [3]:
#定义一系列物理量
grav_Accl = 9.80665; #地表重力加速度(单位: m/s¹)

In [4]:
#以竖直向上为正方向, 计算以5m/s初速度竖直上抛运动的
#位移(单位: m)与时间(单位: s)的关系
displ_vert_prjt = nppn.Polynomial([0, 5, -grav_Accl / 2]); 
displ_vert_prjt

Polynomial([ 0.      ,  5.      , -4.903325], domain=[-1,  1], window=[-1,  1])

### 使用零点列表构造多项式
用法: 
```python
px = np.polynomial.Polynomial.fromroots(iter_root)
```
* `iter_root` 包含多项式零点的迭代器
    * 迭代器结构和元素类型要求与"[使用$x^n$项系数列表构造多项式](#使用$x^n$项系数列表构造多项式)"所用的迭代器相同
* 构造的多项式为$$\prod \limits_{i = 0}^{n - 1} (x - \mathrm{iter\_root}[i]),~\mathrm{其中} n = len(\mathrm{iter\_root})$$
    * 所构造多项式的最高次项系数为1
    * 多项式次数为构造期间使用的迭代器长度
* 迭代器中可以包含重复的元素, 同一元素重复$k$次, 表示所构造的多项式中对应的零点为$k$重零点
* 元素的顺序对所构造多项式的结果无影响

In [5]:
#构造具有零点的高阶多项式
import random; 
import operator as oper; 
from functools import reduce; 
#随机生成10个零点, 均为-10≤x≤10的整数
poly_root = np.random.randint(-10, 10, size=10); 
print(poly_root); 
#利用零点构造10阶多项式
poly_High_Deg = nppn.Polynomial.fromroots(poly_root); 
#利用零点构造多项式的每个因式
poly_Factor = [nppn.Polynomial([-a, 1]) for a in poly_root]; 
#将所有因式连乘(不能使用np.prod)
poly_Expand = reduce(oper.mul, poly_Factor); 
print(poly_High_Deg == poly_Expand)
poly_High_Deg

[ 5 -9 -8  0 -9  7  2  1 -6  6]
True


Polynomial([ 0.000000e+00, -1.632960e+06,  2.442312e+06, -6.782040e+05,
       -1.909980e+05,  5.458700e+04,  6.733000e+03, -1.354000e+03,
       -1.280000e+02,  1.100000e+01,  1.000000e+00], domain=[-1.,  1.], window=[-1.,  1.])

## `numpy.polynomial.Polynomial`对象的调用和修改

### 计算多项式的值
* `Polynomial`对象可作为一元函数使用, 当`np.polynomial.Polynomial`被挂载至`poly`时, 
    ```python
    poly(x)
    ```
    返回多项式`poly`在自变量为`x`时的值
    * 对于由$x^n$项线性组合构造的多项式, 求其在标量$x$下的值的过程, 通过秦九韶-Horner算法实现
    > 可代入`sympy.Symbol("x")`验证(需要`import sympy`)
    * 支持向量化运算. 如果`x`是`list`, `tuple`, `range`, `np.ndarray`等类型, 函数将对`x`中的每个元素求多项式的值, 并返回同型的`np.ndarray`对象. 
    * 不能用于求一元多项式在矩阵$\mathbf{x}$下的值 (即使将`np.ndarray`转换为`np.matrix`对象)

In [6]:
#使用sympy中的符号对象验证Polynomial对象求值过程中采用的算法
import sympy; 
displ_vert_prjt(sympy.Symbol('x'))

1.0*x*(5.0 - 4.903325*x)

In [7]:
coeff_arb = [sympy.Symbol(f'a_{i}') for i in range(6)]; 
nppn.Polynomial(coeff_arb)(sympy.Symbol('x'))

a_0 + 1.0*x*(a_1 + 1.0*x*(a_2 + 1.0*x*(a_3 + 1.0*x*(a_4 + 1.0*a_5*x))))

In [8]:
#计算竖直上抛的物体自释放后0.5s末至3s末间每隔0.5s的位移
print(displ_vert_prjt(np.arange(0.5, 3.00001, 0.5)));

[  1.27416875   0.096675    -3.53248125  -9.6133     -18.14578125
 -29.129925  ]


### 多项式的四则运算
通过对Python数学运算符的重载, `Polynomial`对象支持与数学上的多项式相似的运算
* 多项式与一个常数相加
    * 返回`Polynomial`对象, 常数项为参与运算的常数与多项式常数项之和, 其他项系数保持不变; 
* 多项式与一个常数相乘
    * 返回`Polynomial`对象, 各项系数为参与运算的常数与多项式对应次项系数之积; 
* 两个多项式的加, 减, 乘法; 
* 两个多项式的商(使用整除运算符`//`), 余数(使用取余运算符`%`)
    * 整除运算的返回结果是**仅含有常数项的`Polymal`对象**, 而不是一个数值; 
    * 任意阶的`Polymal`对象, 与任何数值都不相等

In [9]:
import operator as oper; 
leisure = nppn.Polynomial([8, 5, 5]); 
supression = nppn.Polynomial([9, 9, 6]); 
#和, 差, 积
print(supression + 1, supression * (-1)); 
[print(op(leisure, supression), end = "\x20") 
     for op in [oper.add, oper.sub, oper.mul]
]; print();
#商和余数
quot, rmn = [op(leisure, supression) for op in [oper.floordiv, oper.mod]]; 
print(quot, rmn, leisure == supression * quot + rmn); 

poly([10.  9.  6.]) poly([-9. -9. -6.])
poly([17. 14. 11.]) poly([-1. -4. -1.]) poly([ 72. 117. 138.  75.  30.]) 
poly([0.83333333]) poly([ 0.5 -2.5]) True


### 多项式的特殊运算
|运算|用法|备注|
|:-|:-|:-|
|自然数指数幂|`p ** n`|`p` 参与运算的多项式<br>`n` 幂指数, 整型对象, 要求$n \ge 0$<br>返回`Polynomial`对象|
|复合运算|`p(q)`|`p` 外层多项式<br>`q` 内层多项式<br>返回`Polynomial`对象, 仍可作为一元函数<br>使用, 表示参与运算的两个一元多项式的复合<br>函数|
|获取多项式阶数|`p.degree()`|各项的**最高幂次**中的最大值|
|截断|`p.cutdeg(n)`|`p` 参与截断的多项式<br>`n` 截断所得新多项式的阶数<br>返回`Polynomial`对象; 结果为`p`中不高于<br>`n`阶的所有项之和|
|导数|`p.deriv(n)`|`p` 参与求导运算的多项式<br>`n` 导数的阶数, 缺省时为1<br>返回`Polynomial`对象; 求导结果为常函数<br>时, 返回的`Polymal`对象仅含有常数项|
|以0为下限的<br>积分上限函数|`p.integ()`|`p` 参与积分上限函数构造的多项式<br>返回`Polynomial`对象, 其一阶导数为`p`, <br>常数项为`0`|
|累次积分|`p.integ(n)`|`p` 参与累次积分的多项式<br>`n` 对多项式迭代使用`integ()`方法的次数|
|求$\mathbb{C}$上的全部<br>零点(包括重根)|`p.roots()`|`p` 参与求根的多项式<br>返回`np.ndarray`对象, 其`shape`属性<br>值为`(p.degree(), )` ($k$重根记作$k$个根)<br>同一对共轭复根在结果中相邻, 但无法确保<br>所有根按照模长降序排列|

In [10]:
#利用二项式定理构造杨辉-Pascal三角前5行
[print(nppn.Polynomial([1, 1]) ** n) for n in range(5)]
#复合运算不满足交换律
print(leisure, supression); print(leisure(supression), supression(leisure));
#截断至特定阶数
leisure(supression).cutdeg(3)

poly([1.])
poly([1. 1.])
poly([1. 2. 1.])
poly([1. 3. 3. 1.])
poly([1. 4. 6. 4. 1.])
poly([8. 5. 5.]) poly([9. 9. 6.])
poly([458. 855. 975. 540. 180.]) poly([465. 525. 675. 300. 150.])


Polynomial([458., 855., 975., 540.], domain=[-1.,  1.], window=[-1.,  1.])

In [11]:
#导数
[print(supression.deriv(n), end="\x20") 
    for n in range(supression.degree() + 2)
]; print(); 
#积分(常数项恒为0)
[print(supression.integ(n), end="\x20") 
    for n in range(supression.degree() + 2)
]; print(); 

poly([9. 9. 6.]) poly([ 9. 12.]) poly([12.]) poly([0.]) 
poly([9. 9. 6.]) poly([0.  9.  4.5 2. ]) poly([0.  0.  4.5 1.5 0.5]) poly([0.    0.    0.    1.5   0.375 0.1  ]) 


In [12]:
#多项式的复根
#11个系数确定一个10阶多项式, 共计10个复根(k重根记作k个根)
poly_root = nppn.Polynomial(np.random.randint(-10, 11, size=11)).roots(); 
print(poly_root.astype(complex64), np.abs(poly_root), sep="\n")

[-0.8141657 -0.3632193j  -0.8141657 +0.3632193j  -0.39072198-0.47495428j
 -0.39072198+0.47495428j -0.07910223-1.0595082j  -0.07910223+1.0595082j
  0.78723335-0.34982422j  0.78723335+0.34982422j  0.882402  +0.j        ]
[0.89151222 0.89151222 0.61501644 0.61501644 1.06245699 1.06245699
 0.86146    0.86146    0.88240198]
